# 04 - Fine-tuning IndoBERTweet models (`indolem/indobertweet-base-uncased`, `fathan/ijelid-ft-indojave-indobertweet`) with class-weighted loss
> **Universitas Indonesia**
> - Eduardus Tjitrahardja
> - Ikhlasul Akmal Hanif
> - Rahmat Bryan Naufal

## Load Libraries

In [ ]:
DATA_DIR = "../data"

In [77]:
import pandas as pd
import numpy as np
import re

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [78]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())

True


## Load and Clean Data

In [79]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # remove hashtags
    # text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE\s+(\w+)\]', r'\1', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # fix 3
    text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9%]', ' ', text)
    # fix unspaced words
    new_text = []
    for word in text.split():
        # weird word
        if word in ["BEiqnH5xQr", "JN18A"]:
            continue
        # fix shorten names
        elif word == "JK":
            new_text.append("jusuf kalla")
        elif word.lower() == "jkw":
            new_text.append("jokowi")
        elif word.lower() in ["bowo", "ayahbowo"]:
            new_text.append("prabowo")
        elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
            new_text.append("pabrowo gibran")
        elif word == "MasGBRAN":
            new_text.append("gibran")
        elif word == "SBY":
            new_text.append("susilo bambang yudhoyono")
        elif word == "AHY":
            new_text.append("agnes haryani")
        # keep as is
        elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
            new_text.append(word)
        else:
            # fix unspaced words
            new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
    text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

def normalize_word(text):
    map = {
        "sj": "saja",
        "yg": "yang",
        "dgn": "dengan",
        "dg": "dengan",
        "dpt": "dapat",
        "dptkan": "dapatkan",
        "hnya": "hanya",
        "skrng": "sekarang",
        "uu": "undang undang",
        "asn": "aparatur sipil negara",
        "ordal": "orang dalam",
        "mrk": "mereka",
        "rmh": "rumah",
        "kpk": "komisi pemberantasan korupsi",
        "blg": "bilang",
        "tuch": "tuh",
        "umkm": "usaha mikro kecil menengah",
        "jis": "jakarta international stadium",
        "jkt": "jakarta",
        "spt": "seperti",
        "tps": "tempat pemungutan suara",
        "alkes": "alat kesehatan",
        "bowo": "prabowo",
        "org": "orang",
        "pemda": "pemerintah daerah",
        "pemkot": "pemerintah kota",
        "pemprov": "pemerintah provinsi",
        "pemkab": "pemerintah kabupaten",
        "pemilu": "pemilihan umum",
        "pilkada": "pemilihan kepala daerah",
        "pilpres": "pemilihan presiden",
        "pilgub": "pemilihan gubernur",
        "pilbup": "pemilihan bupati",
        "pilwali": "pemilihan wali kota",
        "pilleg": "pemilihan legislatif",
        "pilpres": "pemilihan presiden",
        "ajg" : "bajingan",
        "anjg" : "banjingan",
        "anjir" : "banjingan",
        "anjay" : "banjingan",
        "anjayy" : "banjingan",
        "anjayyy" : "banjingan",
        "ikn" : "ibu kota negara",
    }

    text = text.replace("UU ITE", "Undang Undang Informasi dan Transaksi Elektronik")
    text = text.replace("kitamasbowo dangbran", "kita pilih prabowo dan gibran")
    text = text.replace("PDI P", "PDIP")
    text = text.replace("PDI Perjuangan", "PDIP")
    text = re.sub("alam ganjar", 'anaknya ganjar', text, flags=re.IGNORECASE)
    text = re.sub("%", 'persen', text, flags=re.IGNORECASE)


    party_names = [
        "pkb",
        "gerindra",
        "pdi",
        "pdip",
        "golkar",
        "nasdem",
        "gelora",
        "pks",
        "pkn",
        "hanura",
        "pan",
        "pbb",
        "demokrat",
        "psi",
        "perindo",
        "ppp",
        "ummat",
        "pna",
        "gabthat",
        "pda",
        "sira"
    ]

    stop_words = [
        "dekade08",
        "tempodotco",
        "md"
    ]

    new_text = []
    for word in text.split():
        if word.lower() in map:
            new_text.append(map[word.lower()])
        elif word.lower() in party_names:
            new_text.append("Partai Politik")
        elif word.lower() in stop_words:
            continue
        else:
            new_text.append(word)

    text = ' '.join(new_text)
    return text

def drop_duplicated(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

In [80]:
# To remove inconsistent training data
train_fix_df = pd.read_csv(f'{DATA_DIR}/train_fix - train.csv')

train_fix_df.dropna(subset=["text"], inplace=True)
# get index that is true in is_false column
false_idx = train_fix_df['is_false'] == 1
print(train_fix_df.shape)

(5000, 6)


In [82]:
train = pd.read_csv(f'{DATA_DIR}/train_cleaned_v3.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv', sep=";")
train["text"] = train["text"]
train = train[~false_idx].copy()
print("Train size: ", train.shape)
test['Text'] = test['Text'].apply(remove_noise)
test['Text'] = test['Text'].apply(normalize_word)
test.rename(columns={"Text": "text"}, inplace=True)
train.head()

/tmp/ipykernel_36/1007048891.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  train = train[~false_idx].copy()


Train size:  (4116, 2)


,text,label
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik
4,Anies Baswedan Harap aparatur sipil negara ter...,Politik


In [83]:
# Create a label mapping
label_mapping = {
    'Politik': 0,
    'Sosial Budaya': 1,
    'Ekonomi': 2,
    'Pertahanan dan Keamanan': 3,
    'Ideologi': 4,
    'Sumber Daya Alam': 5,
    'Demografi': 6,
    'Geografi': 7
}

inv_label_mapping = {v: k for k, v in label_mapping.items()}

# Apply the label mapping
train['label'] = train['label'].map(label_mapping)

In [84]:
# Split the DataFrame into training and validation sets
train_df, val_df = train_test_split(train[['text','label']],
                                    test_size=0.2, stratify=train['label'], random_state=42)
train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

In [85]:
import numpy as np
from collections import Counter

# Assuming 'dataset' is your dataset and 'train' split contains the labels
labels = dataset['train']['label']
class_counts = Counter(labels)
total_samples = len(labels)

# Calculate weights for each class
class_weights = {cls: total_samples/count for cls, count in class_counts.items()}

# Normalize weights to sum to 1
sum_weights = sum(class_weights.values())
class_weights = {cls: weight/sum_weights for cls, weight in class_weights.items()}

# Convert weights to a list
class_weight_list = [class_weights[i] for i in range(len(class_weights))]

class_weight_list

[0.0036681907288896695,
 0.02701343106308154,
 0.03941089942536534,
 0.04038641673787438,
 0.04141145269568845,
 0.07998094295147672,
 0.18541036775115058,
 0.5827182986464733]

## Tokenize Data and Fine-tune `indolem/indobertweet-base-uncased` Model

In [86]:
# Define the model name
model_name = "indolem/indobertweet-base-uncased"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define a data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/3292 [00:00<?, ? examples/s]

Map:   0%|          | 0/824 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobertweet-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [87]:
train.label.unique().tolist()

[5, 0, 6, 3, 4, 2, 1, 7]

In [88]:
import torch
import gc


# Mengosongkan cache CUDA
torch.cuda.empty_cache()

# Mengumpulkan garbage
gc.collect()

# Memeriksa penggunaan memori CUDA
print(f"Memory Reserved: {torch.cuda.memory_reserved()} bytes")
print(f"Memory Allocated: {torch.cuda.memory_allocated()} bytes")
print(f"Max Memory Reserved: {torch.cuda.max_memory_reserved()} bytes")

# Restart kernel/environment jika diperlukan

Memory Reserved: 3200253952 bytes
Memory Allocated: 2677132800 bytes
Max Memory Reserved: 10695475200 bytes


In [89]:
import torch
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import balanced_accuracy_score

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Convert class weights to a tensor
        class_weights_tensor = torch.tensor(class_weight_list).to(model.parameters().__next__().device)

        # Compute weighted cross-entropy loss
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Define training arguments with evaluation and saving steps
training_args = TrainingArguments(
    output_dir=f"./{model_name}_results",
    evaluation_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)
    

# Use the custom trainer for training
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

# Train the model
trainer.train()

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss
100,1.693200,1.658641
200,1.184300,1.337984
300,1.154600,1.289874
400,0.819600,1.263007
500,0.745000,1.251455


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead u

TrainOutput(global_step=515, training_loss=1.213648954872946, metrics={'train_runtime': 934.6161, 'train_samples_per_second': 17.612, 'train_steps_per_second': 0.551, 'total_flos': 4331041278689280.0, 'train_loss': 1.213648954872946, 'epoch': 5.0})

In [90]:
# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

# Clear CUDA cache if necessary
torch.cuda.empty_cache()

Balanced Accuracy: 0.5374499876032321


In [91]:
test_dataset = Dataset.from_pandas(test)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
predictions = trainer.predict(tokenized_test)

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)

test['Kelas'] = preds
test['Kelas'] = test['Kelas'].apply(lambda x: inv_label_mapping[x])
test.head()

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

,IDText,text,Kelas
0,TXT0001,Lu mau orang 2 pro demokrasi di negara ini bis...,Ideologi
1,TXT0002,Prabowo ditanya soal hutang luar negeri dia me...,Pertahanan dan Keamanan
2,TXT0003,kiki daliyo Ganjar Pranowo itulah beliau sosok...,Ideologi
3,TXT0004,Prabowo Gibran yang bisa melakukan itu semua d...,Politik
4,TXT0005,Lah justru yang gak nyambung junjungan elu aom...,Sosial Budaya


In [96]:
test.to_csv(f'{DATA_DIR}/submission_indobert_edu.csv', index=False)

## Tokenize Data and Fine-tune `fathan/ijelid-ft-indojave-indobertweet` Model

In [99]:
# Define the model name
model_name = "fathan/ijelid-ft-indojave-indobertweet"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define a data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/738k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/3292 [00:00<?, ? examples/s]

Map:   0%|          | 0/824 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at fathan/ijelid-ft-indojave-indobertweet and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at fathan/ijelid-ft-indojave-indobertweet and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([7, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([7]) in the checkpoint and torch.Size([8]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [100]:
train.label.unique().tolist()

[5, 0, 6, 3, 4, 2, 1, 7]

In [101]:
import torch
import gc


# Mengosongkan cache CUDA
torch.cuda.empty_cache()

# Mengumpulkan garbage
gc.collect()

# Memeriksa penggunaan memori CUDA
print(f"Memory Reserved: {torch.cuda.memory_reserved()} bytes")
print(f"Memory Allocated: {torch.cuda.memory_allocated()} bytes")
print(f"Max Memory Reserved: {torch.cuda.max_memory_reserved()} bytes")

# Restart kernel/environment jika diperlukan

Memory Reserved: 3208642560 bytes
Memory Allocated: 1347185664 bytes
Max Memory Reserved: 10695475200 bytes


In [102]:
import torch
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import balanced_accuracy_score

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Convert class weights to a tensor
        class_weights_tensor = torch.tensor(class_weight_list).to(model.parameters().__next__().device)

        # Compute weighted cross-entropy loss
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Define training arguments with evaluation and saving steps
training_args = TrainingArguments(
    output_dir=f"./{model_name}_results",
    evaluation_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)
    

# Use the custom trainer for training
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

# Train the model
trainer.train()

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss
100,1.776900,1.697050
200,1.175800,1.362278
300,1.171000,1.324110
400,0.792800,1.284857
500,0.754600,1.273642


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead u

TrainOutput(global_step=515, training_loss=1.236064040545121, metrics={'train_runtime': 927.7007, 'train_samples_per_second': 17.743, 'train_steps_per_second': 0.555, 'total_flos': 4331041278689280.0, 'train_loss': 1.236064040545121, 'epoch': 5.0})

In [103]:
# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

# Clear CUDA cache if necessary
torch.cuda.empty_cache()

Balanced Accuracy: 0.5530076057993473


In [104]:
test_dataset = Dataset.from_pandas(test)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
predictions = trainer.predict(tokenized_test)

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)

test['Kelas'] = preds
test['Kelas'] = test['Kelas'].apply(lambda x: inv_label_mapping[x])
test.head()

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

,IDText,text,Kelas
0,TXT0001,Lu mau orang 2 pro demokrasi di negara ini bis...,Politik
1,TXT0002,Prabowo ditanya soal hutang luar negeri dia me...,Pertahanan dan Keamanan
2,TXT0003,kiki daliyo Ganjar Pranowo itulah beliau sosok...,Ideologi
3,TXT0004,Prabowo Gibran yang bisa melakukan itu semua d...,Politik
4,TXT0005,Lah justru yang gak nyambung junjungan elu aom...,Ideologi


In [70]:
test.to_csv(f'{DATA_DIR}/submission_ijelid.csv', index=False)